Make sure the right schema is used

In [0]:
USE CATALOG sac;

USE SCHEMA customer_service;

# Create bronze tables

In [0]:
CREATE TABLE IF NOT EXISTS customer_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS churn_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS log_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS ticket_bronze (
        ticket_id STRING,
        customer_id STRING,
        timestamp_created STRING,
        timestamp_closed STRING,
        subject STRING,
        description STRING,
        category STRING,
        priority STRING,
        channel STRING,
        status STRING,
        solved_in_hours STRING,
        log_id STRING,
        technical_issue_type STRING,
        ingestion_time TIMESTAMP
    );

In [0]:
CREATE TABLE IF NOT EXISTS agent_bronze;

In [0]:
CREATE TABLE IF NOT EXISTS chat_bronze (
        session_id STRING,
        customer_id STRING,
        agent_id STRING,
        timestamp_start STRING,
        timestamp_end STRING,
        messages STRING,
        resolution_status STRING,
        log_id STRING,
        chat_reason STRING,
        ingestion_time TIMESTAMP
    )

# Fill tables

In [0]:
COPY INTO customer_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/customer_profiles/'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true', 'multiLine' = 'true') COPY_OPTIONS ('mergeSchema' = 'true');

In [0]:
COPY INTO churn_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/churn_labels'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
COPY INTO log_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/connection_quality_logs'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
COPY INTO agent_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    '/Volumes/workspace/cs_stream/cs_stream/agent_profiles'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%python

%pip install pandas openpyxl


In [0]:
%python

import pandas
import openpyxl
from pyspark.sql.functions import current_timestamp

In [0]:
%skip
COPY INTO ticket_bronze FROM (
    SELECT
        *,
        current_timestamp() AS ingestion_time FROM
    './data/support_ticket.xlsx'
) FILEFORMAT = CSV FORMAT_OPTIONS ('header' = 'true') COPY_OPTIONS ('mergeSchema' = 'true')

In [0]:
%python

df_ticket = pandas.read_excel("/Workspace/Users/nina.merkt@abat.de/databricks_demo/data/support_tickets/support_ticket.xlsx", engine='openpyxl')

print('Shape: ', df_ticket.shape)
print('Columns: ', df_ticket.columns)
print('Data Types: ', df_ticket.dtypes)

df_ticket = df_ticket.astype(str)
spark_df_ticket = spark.createDataFrame(df_ticket)

spark_df_ticket = spark_df_ticket.withColumn("ingestion_time", current_timestamp())

spark_df_ticket.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("ticket_bronze")

In [0]:
%python

df_chat = pandas.read_excel("/Workspace/Users/nina.merkt@abat.de/databricks_demo/data/chat_transcripts/chat_transcript.xlsx", engine='openpyxl')

print('Shape: ', df_chat.shape)
print('Columns: ', df_chat.columns)
print('Data Types: ', df_chat.dtypes)
from pyspark.sql.functions import current_timestamp

df_chat = df_chat.astype(str)
spark_df_chat = spark.createDataFrame(df_chat)

spark_df_chat = spark_df_chat.withColumn("ingestion_time", current_timestamp())

spark_df_chat.write \
    .format("delta") \
    .mode("append") \
    .saveAsTable("chat_bronze")